In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
input_gnomad = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/sas_filtered_normalized_final/gnomad_chr22_sas_normalized.vcf.gz"
reference_fasta = "/content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa"
output_dir = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/pure_sas_filtered_normalized_FINAL_VERSION"

In [5]:
!pip install cyvcf2
!apt-get update
!apt-get install -y bcftools tabix

from cyvcf2 import VCF
import subprocess
import os

# Ancestries to check == 0 (all except sas)
OTHER_ANCESTRIES = ["afr", "ami", "amr", "asj", "eas", "fin", "mid", "nfe", "remaining"]

# Build the bcftools filter expression
zero_conditions = " && ".join([f"INFO/AC_joint_{anc} = 0" for anc in OTHER_ANCESTRIES])
filter_expr = f'INFO/AC_joint_sas >= 1 && {zero_conditions}'

output_file = f"{output_dir}/gnomad_chr22_pure_sas.vcf.gz"

filter_cmd = f"""
bcftools view \
    -i '{filter_expr}' \
    -O z \
    -o {output_file} \
    {input_gnomad}
"""

print("Filtering for SAS-exclusive variants...")
print(f"Filter: {filter_expr}\n")

result = subprocess.run(filter_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("\n❌ FILTERING FAILED!")
    print(result.stderr)
    raise Exception("Pure SAS filtering failed")

# Index the output
subprocess.run(f"tabix -p vcf {output_file}", shell=True)

# Count results
count_cmd = f"bcftools view -H {output_file} | wc -l"
result = subprocess.run(count_cmd, shell=True, capture_output=True, text=True)
filtered_count = result.stdout.strip()

print(f"✓ Filtering complete!")
print(f"  SAS-exclusive variants: {filtered_count}")

filtered_size = os.path.getsize(output_file) / (1024**3)
print(f"  File size: {filtered_size:.2f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.9 MB/s eta 0:00:00
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,921 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,810 kB]
Get:11 https://ppa.launchpadcon

In [6]:
def quality_check_pure_sas(vcf_path: str, n_violations: int = 5):
    OTHER_ANCESTRIES = ["afr", "ami", "amr", "asj", "eas", "fin", "mid", "nfe", "remaining"]

    vcf = VCF(vcf_path)

    total_variants        = 0
    sas_zero_violations   = 0  # variants where AC_joint_sas == 0 (should never happen)
    other_nonzero_violations = 0  # variants where any other ancestry AC > 0 (should never happen)
    violation_examples    = []

    print("Running quality checks on SAS-exclusive VCF...")
    print("=" * 80)

    for variant in vcf:
        total_variants += 1

        ac_sas = variant.INFO.get("AC_joint_sas") or 0

        # CHECK 1: SAS must be >= 1
        if ac_sas < 1:
            sas_zero_violations += 1

        # CHECK 2: All other ancestries must be 0
        offending = []
        for anc in OTHER_ANCESTRIES:
            ac = variant.INFO.get(f"AC_joint_{anc}") or 0
            if ac > 0:
                offending.append(f"{anc}={ac}")

        if offending:
            other_nonzero_violations += 1
            if len(violation_examples) < n_violations:
                violation_examples.append({
                    "variant": f"{variant.CHROM}:{variant.POS} {variant.REF}>{variant.ALT[0]}",
                    "AC_joint_sas": ac_sas,
                    "offenders": ", ".join(offending)
                })

    # ---- REPORT ----
    print(f"Total variants checked       : {total_variants}")
    print(f"CHECK 1 - AC_joint_sas >= 1  : ", end="")
    if sas_zero_violations == 0:
        print(f"✅ PASSED (0 violations)")
    else:
        print(f"❌ FAILED ({sas_zero_violations} violations)")

    print(f"CHECK 2 - All others == 0    : ", end="")
    if other_nonzero_violations == 0:
        print(f"✅ PASSED (0 violations)")
    else:
        print(f"❌ FAILED ({other_nonzero_violations} violations)")

    if violation_examples:
        print(f"\nExample violations (up to {n_violations}):")
        print("-" * 80)
        for v in violation_examples:
            print(f"  Variant      : {v['variant']}")
            print(f"  AC_joint_sas : {v['AC_joint_sas']}")
            print(f"  Offenders    : {v['offenders']}")
            print()

    print("=" * 80)
    passed = (sas_zero_violations == 0 and other_nonzero_violations == 0)
    print(f"OVERALL: {'✅ ALL CHECKS PASSED' if passed else '❌ CHECKS FAILED'}")

    return passed

# Run it
quality_check_pure_sas(output_file)

Running quality checks on SAS-exclusive VCF...
Total variants checked       : 643676
CHECK 1 - AC_joint_sas >= 1  : ✅ PASSED (0 violations)
CHECK 2 - All others == 0    : ✅ PASSED (0 violations)
OVERALL: ✅ ALL CHECKS PASSED


True

In [7]:
# ------------------------------------------------ #

In [9]:
# ============================================================
# CONFIGURATION — change these to run for any chromosome
# ============================================================
CHR = "21"
input_gnomad    = f"/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/sas_filtered_normalized_final/gnomad_chr{CHR}_sas_normalized.vcf.gz"
reference_fasta = "/content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa"
output_dir      = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/pure_sas_filtered_normalized_FINAL_VERSION"
output_file     = f"{output_dir}/gnomad_chr{CHR}_pure_sas.vcf.gz"

OTHER_ANCESTRIES = ["afr", "ami", "amr", "asj", "eas", "fin", "mid", "nfe", "remaining"]

# ============================================================
# INSTALL & IMPORTS
# ============================================================
!pip install cyvcf2 -q
!apt-get update -q
!apt-get install -y bcftools tabix -q

from cyvcf2 import VCF
import subprocess
import os

# ============================================================
# STEP 1 — FILTER
# ============================================================
os.makedirs(output_dir, exist_ok=True)

zero_conditions = " && ".join([f"INFO/AC_joint_{anc} = 0" for anc in OTHER_ANCESTRIES])
filter_expr     = f'INFO/AC_joint_sas >= 1 && {zero_conditions}'

filter_cmd = f"""
bcftools view \
    -i '{filter_expr}' \
    -O z \
    -o {output_file} \
    {input_gnomad}
"""

print(f"{'='*60}")
print(f"  Chromosome  : chr{CHR}")
print(f"  Input       : {input_gnomad}")
print(f"  Output      : {output_file}")
print(f"  Filter      : {filter_expr}")
print(f"{'='*60}\n")

print("Filtering for SAS-exclusive variants...")
result = subprocess.run(filter_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ FILTERING FAILED!")
    print(result.stderr)
    raise Exception("Pure SAS filtering failed")

print("✅ Filtering complete!")

# ---- Index ----
print("Indexing output file...")
result = subprocess.run(f"tabix -p vcf {output_file}", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("❌ INDEXING FAILED!")
    print(result.stderr)
    raise Exception("Indexing failed")
print("✅ Indexing complete!")

# ---- Count ----
count_result = subprocess.run(f"bcftools view -H {output_file} | wc -l", shell=True, capture_output=True, text=True)
filtered_count = count_result.stdout.strip()
filtered_size  = os.path.getsize(output_file) / (1024**3)

print(f"\n  SAS-exclusive variants : {filtered_count}")
print(f"  File size              : {filtered_size:.4f} GB")


# ============================================================
# STEP 2 — QUALITY CHECKS
# ============================================================
def quality_check_pure_sas(vcf_path: str, n_violations: int = 5):

    vcf = VCF(vcf_path)

    total_variants           = 0
    sas_zero_violations      = 0
    other_nonzero_violations = 0
    violation_examples       = []

    print(f"\n{'='*60}")
    print(f"  QUALITY CHECKS — chr{CHR}")
    print(f"{'='*60}")

    for variant in vcf:
        total_variants += 1

        ac_sas = variant.INFO.get("AC_joint_sas") or 0

        # CHECK 1: SAS must be >= 1
        if ac_sas < 1:
            sas_zero_violations += 1

        # CHECK 2: All other ancestries must be 0
        offending = []
        for anc in OTHER_ANCESTRIES:
            ac = variant.INFO.get(f"AC_joint_{anc}") or 0
            if ac > 0:
                offending.append(f"{anc}={ac}")

        if offending:
            other_nonzero_violations += 1
            if len(violation_examples) < n_violations:
                violation_examples.append({
                    "variant"     : f"{variant.CHROM}:{variant.POS} {variant.REF}>{variant.ALT[0]}",
                    "AC_joint_sas": ac_sas,
                    "offenders"   : ", ".join(offending)
                })

    # ---- Report ----
    print(f"  Total variants checked       : {total_variants}")
    print(f"  CHECK 1 - AC_joint_sas >= 1  : ", end="")
    if sas_zero_violations == 0:
        print(f"✅ PASSED (0 violations)")
    else:
        print(f"❌ FAILED ({sas_zero_violations} violations)")

    print(f"  CHECK 2 - All others == 0    : ", end="")
    if other_nonzero_violations == 0:
        print(f"✅ PASSED (0 violations)")
    else:
        print(f"❌ FAILED ({other_nonzero_violations} violations)")

    if violation_examples:
        print(f"\n  Example violations (up to {n_violations}):")
        print("-" * 60)
        for v in violation_examples:
            print(f"    Variant      : {v['variant']}")
            print(f"    AC_joint_sas : {v['AC_joint_sas']}")
            print(f"    Offenders    : {v['offenders']}")
            print()

    print("=" * 60)
    passed = (sas_zero_violations == 0 and other_nonzero_violations == 0)
    print(f"  OVERALL: {'✅ ALL CHECKS PASSED' if passed else '❌ CHECKS FAILED'}")
    print("=" * 60)

    return passed


quality_check_pure_sas(output_file)

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
bcftools is already the newest version (1.13-1).
tabix is already the newest version (1.13+ds-2build1).
0 upgraded

True

In [10]:
# ============================================================
# CONFIGURATION — change these to run for any chromosome
# ============================================================
CHR = "20"
input_gnomad    = f"/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/sas_filtered_normalized_final/gnomad_chr{CHR}_sas_normalized.vcf.gz"
reference_fasta = "/content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa"
output_dir      = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/pure_sas_filtered_normalized_FINAL_VERSION"
output_file     = f"{output_dir}/gnomad_chr{CHR}_pure_sas.vcf.gz"

OTHER_ANCESTRIES = ["afr", "ami", "amr", "asj", "eas", "fin", "mid", "nfe", "remaining"]

# ============================================================
# INSTALL & IMPORTS
# ============================================================
!pip install cyvcf2 -q
!apt-get update -q
!apt-get install -y bcftools tabix -q

from cyvcf2 import VCF
import subprocess
import os

# ============================================================
# STEP 1 — FILTER
# ============================================================
os.makedirs(output_dir, exist_ok=True)

zero_conditions = " && ".join([f"INFO/AC_joint_{anc} = 0" for anc in OTHER_ANCESTRIES])
filter_expr     = f'INFO/AC_joint_sas >= 1 && {zero_conditions}'

filter_cmd = f"""
bcftools view \
    -i '{filter_expr}' \
    -O z \
    -o {output_file} \
    {input_gnomad}
"""

print(f"{'='*60}")
print(f"  Chromosome  : chr{CHR}")
print(f"  Input       : {input_gnomad}")
print(f"  Output      : {output_file}")
print(f"  Filter      : {filter_expr}")
print(f"{'='*60}\n")

print("Filtering for SAS-exclusive variants...")
result = subprocess.run(filter_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ FILTERING FAILED!")
    print(result.stderr)
    raise Exception("Pure SAS filtering failed")

print("✅ Filtering complete!")

# ---- Index ----
print("Indexing output file...")
result = subprocess.run(f"tabix -p vcf {output_file}", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("❌ INDEXING FAILED!")
    print(result.stderr)
    raise Exception("Indexing failed")
print("✅ Indexing complete!")

# ---- Count ----
count_result = subprocess.run(f"bcftools view -H {output_file} | wc -l", shell=True, capture_output=True, text=True)
filtered_count = count_result.stdout.strip()
filtered_size  = os.path.getsize(output_file) / (1024**3)

print(f"\n  SAS-exclusive variants : {filtered_count}")
print(f"  File size              : {filtered_size:.4f} GB")


# ============================================================
# STEP 2 — QUALITY CHECKS
# ============================================================
def quality_check_pure_sas(vcf_path: str, n_violations: int = 5):

    vcf = VCF(vcf_path)

    total_variants           = 0
    sas_zero_violations      = 0
    other_nonzero_violations = 0
    violation_examples       = []

    print(f"\n{'='*60}")
    print(f"  QUALITY CHECKS — chr{CHR}")
    print(f"{'='*60}")

    for variant in vcf:
        total_variants += 1

        ac_sas = variant.INFO.get("AC_joint_sas") or 0

        # CHECK 1: SAS must be >= 1
        if ac_sas < 1:
            sas_zero_violations += 1

        # CHECK 2: All other ancestries must be 0
        offending = []
        for anc in OTHER_ANCESTRIES:
            ac = variant.INFO.get(f"AC_joint_{anc}") or 0
            if ac > 0:
                offending.append(f"{anc}={ac}")

        if offending:
            other_nonzero_violations += 1
            if len(violation_examples) < n_violations:
                violation_examples.append({
                    "variant"     : f"{variant.CHROM}:{variant.POS} {variant.REF}>{variant.ALT[0]}",
                    "AC_joint_sas": ac_sas,
                    "offenders"   : ", ".join(offending)
                })

    # ---- Report ----
    print(f"  Total variants checked       : {total_variants}")
    print(f"  CHECK 1 - AC_joint_sas >= 1  : ", end="")
    if sas_zero_violations == 0:
        print(f"✅ PASSED (0 violations)")
    else:
        print(f"❌ FAILED ({sas_zero_violations} violations)")

    print(f"  CHECK 2 - All others == 0    : ", end="")
    if other_nonzero_violations == 0:
        print(f"✅ PASSED (0 violations)")
    else:
        print(f"❌ FAILED ({other_nonzero_violations} violations)")

    if violation_examples:
        print(f"\n  Example violations (up to {n_violations}):")
        print("-" * 60)
        for v in violation_examples:
            print(f"    Variant      : {v['variant']}")
            print(f"    AC_joint_sas : {v['AC_joint_sas']}")
            print(f"    Offenders    : {v['offenders']}")
            print()

    print("=" * 60)
    passed = (sas_zero_violations == 0 and other_nonzero_violations == 0)
    print(f"  OVERALL: {'✅ ALL CHECKS PASSED' if passed else '❌ CHECKS FAILED'}")
    print("=" * 60)

    return passed


quality_check_pure_sas(output_file)

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 129 kB in 1s (139 kB/s)
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
bcftools is already the newest version (1.13-1).
tabix is already the new

True

In [11]:
# ============================================================
# CONFIGURATION — change these to run for any chromosome
# ============================================================
CHR = "19"
input_gnomad    = f"/content/drive/MyDrive/FYP_DATA/DATA/processed_data/gnomad/sas_filtered_normalized_final/gnomad_chr{CHR}_sas_normalized.vcf.gz"
reference_fasta = "/content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa"
output_dir      = "/content/drive/MyDrive/FYP_DATA/DATA/processed_data/pure_sas_filtered_normalized_FINAL_VERSION"
output_file     = f"{output_dir}/gnomad_chr{CHR}_pure_sas.vcf.gz"

OTHER_ANCESTRIES = ["afr", "ami", "amr", "asj", "eas", "fin", "mid", "nfe", "remaining"]

# ============================================================
# INSTALL & IMPORTS
# ============================================================
!pip install cyvcf2 -q
!apt-get update -q
!apt-get install -y bcftools tabix -q

from cyvcf2 import VCF
import subprocess
import os

# ============================================================
# STEP 1 — FILTER
# ============================================================
os.makedirs(output_dir, exist_ok=True)

zero_conditions = " && ".join([f"INFO/AC_joint_{anc} = 0" for anc in OTHER_ANCESTRIES])
filter_expr     = f'INFO/AC_joint_sas >= 1 && {zero_conditions}'

filter_cmd = f"""
bcftools view \
    -i '{filter_expr}' \
    -O z \
    -o {output_file} \
    {input_gnomad}
"""

print(f"{'='*60}")
print(f"  Chromosome  : chr{CHR}")
print(f"  Input       : {input_gnomad}")
print(f"  Output      : {output_file}")
print(f"  Filter      : {filter_expr}")
print(f"{'='*60}\n")

print("Filtering for SAS-exclusive variants...")
result = subprocess.run(filter_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ FILTERING FAILED!")
    print(result.stderr)
    raise Exception("Pure SAS filtering failed")

print("✅ Filtering complete!")

# ---- Index ----
print("Indexing output file...")
result = subprocess.run(f"tabix -p vcf {output_file}", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print("❌ INDEXING FAILED!")
    print(result.stderr)
    raise Exception("Indexing failed")
print("✅ Indexing complete!")

# ---- Count ----
count_result = subprocess.run(f"bcftools view -H {output_file} | wc -l", shell=True, capture_output=True, text=True)
filtered_count = count_result.stdout.strip()
filtered_size  = os.path.getsize(output_file) / (1024**3)

print(f"\n  SAS-exclusive variants : {filtered_count}")
print(f"  File size              : {filtered_size:.4f} GB")


# ============================================================
# STEP 2 — QUALITY CHECKS
# ============================================================
def quality_check_pure_sas(vcf_path: str, n_violations: int = 5):

    vcf = VCF(vcf_path)

    total_variants           = 0
    sas_zero_violations      = 0
    other_nonzero_violations = 0
    violation_examples       = []

    print(f"\n{'='*60}")
    print(f"  QUALITY CHECKS — chr{CHR}")
    print(f"{'='*60}")

    for variant in vcf:
        total_variants += 1

        ac_sas = variant.INFO.get("AC_joint_sas") or 0

        # CHECK 1: SAS must be >= 1
        if ac_sas < 1:
            sas_zero_violations += 1

        # CHECK 2: All other ancestries must be 0
        offending = []
        for anc in OTHER_ANCESTRIES:
            ac = variant.INFO.get(f"AC_joint_{anc}") or 0
            if ac > 0:
                offending.append(f"{anc}={ac}")

        if offending:
            other_nonzero_violations += 1
            if len(violation_examples) < n_violations:
                violation_examples.append({
                    "variant"     : f"{variant.CHROM}:{variant.POS} {variant.REF}>{variant.ALT[0]}",
                    "AC_joint_sas": ac_sas,
                    "offenders"   : ", ".join(offending)
                })

    # ---- Report ----
    print(f"  Total variants checked       : {total_variants}")
    print(f"  CHECK 1 - AC_joint_sas >= 1  : ", end="")
    if sas_zero_violations == 0:
        print(f"✅ PASSED (0 violations)")
    else:
        print(f"❌ FAILED ({sas_zero_violations} violations)")

    print(f"  CHECK 2 - All others == 0    : ", end="")
    if other_nonzero_violations == 0:
        print(f"✅ PASSED (0 violations)")
    else:
        print(f"❌ FAILED ({other_nonzero_violations} violations)")

    if violation_examples:
        print(f"\n  Example violations (up to {n_violations}):")
        print("-" * 60)
        for v in violation_examples:
            print(f"    Variant      : {v['variant']}")
            print(f"    AC_joint_sas : {v['AC_joint_sas']}")
            print(f"    Offenders    : {v['offenders']}")
            print()

    print("=" * 60)
    passed = (sas_zero_violations == 0 and other_nonzero_violations == 0)
    print(f"  OVERALL: {'✅ ALL CHECKS PASSED' if passed else '❌ CHECKS FAILED'}")
    print("=" * 60)

    return passed


quality_check_pure_sas(output_file)

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,120 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,613 kB]
Fetched 5,988 kB in 2s (2,556 kB/s)
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it

True